In [26]:
from pygmid import Lookup as lk
import numpy as np
import scipy.constants as sc
import pandas as pd

# read data for GF180MCU NMOS and PMOS device
# the range of channel lengths is 0.28 to 3
# the range for VGS, VDS is 0 to 3.3
n = lk('../../bmurmann_gmoverid/gf180mcuD/simulation/nfet_03v3.mat')
p = lk('../../bmurmann_gmoverid/gf180mcuD/simulation/pfet_03v3.mat')

In [27]:
# #Parameters
# Ceff = 10e-15 #delay stage output capacitance
# n = 4 #the number of VCO stage
# Cb = 2 * n * Ceff #Total stage output capacitance times 2
# Id = 240e-6 #Delay stage's bias current
# x = 1 #Current ratio
# y = 4 #Resistance ratio
# Ich = x * (2 * Id) #Charge pump current
# C1 = 10e-12 #Loop filter capacitor
# N = 16 #Divider ratio

# damping_factor = y / 4 * np.sqrt(x / N) * np.sqrt(C1 / Cb)
# freq_ratio = x * N / (2 * np.pi) * np.sqrt(Cb / C1)


In [28]:
#5T OTA sizing
GBW = 10e6 #OTA bandwidth
CL = 10e-12 #OTA load capacitance
gm_id0 = 12
gm_id1 = 12
gm_id2 = 12
l0 = 0.5
l1 = 0.5
l2 = 0.5

In [31]:
# calculate gm of differential pair
gm1 = 2*np.pi*GBW*CL

# size all transistors
id1 = gm1/gm_id1; id2=id1; id0=2*id1
jd1 = p.lookup('ID_W', GM_ID=gm_id1, L=l1)
w1 = id1/jd1
jd2 = n.lookup('ID_W', GM_ID=gm_id2, L=l2)
w2 = id1/jd2
w0 = 2*w1

# estimate mirror pole
cgg2 = w2*p.lookup('CGG_W', GM_ID=gm_id2, L=l2)
cdd2 = w2*p.lookup('CDD_W', GM_ID=gm_id2, L=l2)
cdd1 = w2*n.lookup('CDD_W', GM_ID=gm_id1, L=l1)
gm2 = gm1
fp2 = gm2/(2*cgg2+cdd1+cdd2)/(2*np.pi)

# estimate phase margin (mirror pole, LHP zero, RHP zero)
phip2 = -np.arctan(GBW/fp2)*180/np.pi
fz2 = 2*fp2
phiz2 = +np.arctan(GBW/fz2)*180/np.pi
cgd1 = w1*n.lookup('CGD_W', GM_ID=gm_id1, L=l1)
fz3 = gm1/cgd1/(2*np.pi)
phiz3 = -np.arctan(GBW/fz3)*180/np.pi
PM = 90 +phip2 +phiz2 +phiz3 

df = pd.DataFrame( [id1/1e-6, gm1/1e-3, fp2, PM], \
                   ['id1 (uA)', 'gm1 (mS)', 'fp2 (Hz)', 'PM (deg)'], columns=['Value']); df.round(2)


,Value
id1 (uA),5.236000e+01
gm1 (mS),6.300000e-01
fp2 (Hz),9.725595e+08
PM (deg),8.965000e+01


In [30]:
# finger the devices
wfing = 5
nf0 = 1+np.floor_divide(w0, wfing)
nf1 = 1+np.floor_divide(w1, wfing)
nf2 = 1+np.floor_divide(w2, wfing)
df = pd.DataFrame( [(id0*1e6, id1*1e6, id2*1e6), (w0, w1, w2), (l0, l1, l2), (nf0, nf1, nf2)], \
                   ['ID (uA)', 'w (um)', 'l (um)', 'nf'], columns=['M0', 'M1', 'M2']); df.round(2)

,M0,M1,M2
ID (uA),104.72,52.36,52.36
w (um),133.03,66.52,19.25
l (um),0.50,0.50,0.50
nf,27.00,14.00,4.00
